# Week 11: Error-Analysis Workshop

# Requirements: pip install numpy pandas

# ⚠️ No API key needed

Eval gives you a *score*; error analysis tells you *which cluster to fix*. This notebook
runs the full loop on a deterministic synthetic run log from the Weeks 6 to 10 artifacts:
**read traces → cluster failures → HLP → pick the top fix → write the CI gate** as a
callable function that fails below a threshold.


## 0. Setup: repo root on the path + seeded RNG


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import numpy as np
import pandas as pd

from zoro import data

SEED = 42
np.random.seed(SEED)
print("imports ok")


## 1. Generate a synthetic run log

In production this log comes from your tracing backend. Here we generate it
deterministically so the workshop is reproducible: 50 traces across three task types,
each marked `correct`/`wrong` with an `error_type` when wrong. 30% are correct on
purpose, error analysis reads the good ones too, for contrast.


In [ ]:
def make_run_log(n=50, seed=42):
    rng = np.random.default_rng(seed)
    error_types = [
        ("retrieval_wrong_doc", "rag"),         # right doc ranked 4th -> answered from wrong doc
        ("field_date_format", "extraction"),    # MM/DD vs DD/MM silently changed meaning
        ("wrong_category", "triage"),           # billing routed to claims
        ("hallucinated_rate", "rag"),           # invented a refund % not in the passage
        ("low_confidence", "triage"),           # answered when it should have declined
        ("schema_invalid_json", "extraction"),  # returned non-JSON
    ]
    weights = [0.28, 0.18, 0.20, 0.14, 0.12, 0.08]
    logs = []
    for i in range(n):
        correct = bool(rng.random() < 0.30)
        if correct:
            logs.append({"trace_id": f"tr-{i:03d}", "task": error_types[i % len(error_types)][1],
                         "verdict": "correct", "error_type": None, "detail": "n/a"})
        else:
            et, task = error_types[rng.choice(len(error_types), p=weights)]
            logs.append({"trace_id": f"tr-{i:03d}", "task": task, "verdict": "wrong",
                         "error_type": et, "detail": f"{et} on {task} input"})
    return logs

run_log = make_run_log(n=50, seed=42)
logs_df = pd.DataFrame(run_log)
print("run log rows:", len(logs_df))
print(logs_df["verdict"].value_counts().to_string())


## 2. Read the traces by hand (don't summarize from memory)

The rule is to look at what the system *actually did*, good and bad. Read the sample
below before grouping.


In [ ]:
good = logs_df[logs_df.verdict == "correct"].head(5)
bad = logs_df[logs_df.verdict == "wrong"].head(8)
print("--- good traces (contrast) ---")
print(good.to_string(index=False))
print("\n--- bad traces (read by hand) ---")
print(bad.to_string(index=False))


## 3. Cluster the failures

Group the wrong rows by `error_type` and count. Frequency matters more than cleverness,
the biggest class is the one to fix.


In [ ]:
wrong = logs_df[logs_df.verdict == "wrong"]
clusters = wrong.groupby("error_type").size().sort_values(ascending=False)
print("failure clusters (count):")
print(clusters.to_string())


## 4. HLP: human-level parity

For each class, ask: *would a competent human, given the same information, have gotten
this right?* `fixable` = the system underperforms a human → fix the system. `upstream` =
even a human would fail → fix the data/intake, not the model.


In [ ]:
HLP_MAP = {
    "retrieval_wrong_doc": ("fixable",  "a human would pick the right doc from the top-5"),
    "field_date_format":   ("fixable",  "a human sees MM/DD vs DD/MM and preserves meaning"),
    "wrong_category":      ("fixable",  "a human routes billing -> billing"),
    "hallucinated_rate":   ("fixable",  "a human would not invent a rate"),
    "low_confidence":      ("fixable",  "a human would escalate instead of guessing"),
    "schema_invalid_json": ("upstream", "both model and human need a schema contract"),
}


In [ ]:
rows = []
for et, count in clusters.items():
    kind, why = HLP_MAP.get(et, ("unknown", ""))
    rows.append({"error_type": et, "count": int(count), "hlp": kind, "why": why})
hlp_df = pd.DataFrame(rows)
print(hlp_df.to_string(index=False))

fixable_total = int(hlp_df[hlp_df.hlp == "fixable"]["count"].sum())
upstream_total = int(hlp_df[hlp_df.hlp == "upstream"]["count"].sum())
print(f"\nfixable (system underperforms human): {fixable_total}   |   upstream (human also fails): {upstream_total}")


## 5. Prioritize: pick the top fix

The largest **fixable** class wins. That is the one change worth making this week, and,
crucially, the class you will add back to the eval set so it can never silently regress.


In [ ]:
fixable = hlp_df[hlp_df.hlp == "fixable"].sort_values("count", ascending=False)
top_fix = fixable.iloc[0]["error_type"] if len(fixable) else None
top_count = int(fixable.iloc[0]["count"]) if len(fixable) else 0
coverage = top_count / max(fixable_total, 1)
print("top fixable class:", top_fix, "| count:", top_count)
print(f"fixing it addresses {coverage:.1%} of fixable failures")


## 6. Write the CI gate

A gate is a **threshold on a number**, not a green checkmark. `ci_gate` is a callable
function: pass a score and a threshold, it prints PASS/FAIL and returns the boolean. Wire
it into CI and a regression blocks the merge.


In [ ]:
def ci_gate(score, threshold=0.9, metric="eval"):
    """The release gate: fail (return False) when a metric is below its threshold."""
    passed = score >= threshold
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {metric} = {score:.3f}  (threshold {threshold})")
    return passed


## 7. Apply the gate to a simulated release

A good artifact passes; a regressed one (extraction below its 0.90 bar) is blocked. The
point is that the gate has *authority*, it is willing to say no.


In [ ]:
scores = {"triage_accuracy": 0.93, "extraction_field_accuracy": 0.88, "groundedness": 0.95}
thresholds = {"triage_accuracy": 0.90, "extraction_field_accuracy": 0.90, "groundedness": 0.95}

results = {k: ci_gate(v, thresholds[k], k) for k, v in scores.items()}
print("\nrelease blocked? ", not all(results.values()))


## 8. Takeaway

Evaluation tells you the current level; error analysis tells you what to do next. The
ratchet that makes it stick is step five of the loop: **add the failure class back to the
eval set**, which is exactly what ZoroEval (notebook 01) exists to hold. This workshop
is the cheapest, highest-leverage exercise in the whole program: fifty cases, read by
hand, categorized, counted, largest class fixed, added to the eval.


In [ ]:
# FINAL numbers: the top fix's coverage of fixable failures, and the gate's pass rate.
print(f"TOP_FIX_COVERAGE={coverage:.3f}")
print(f"GATE_PASS_RATE={sum(results.values())}/{len(results)}")
